In [ ]:
# Example 1: Public API surface — confirm the documented exports exist and have the expected types.
from dataclasses import is_dataclass
from enum import Enum

from detect_installer import Installer, InstallerInfo, UvUpgradeStrategy, detect_installer

assert callable(detect_installer), "detect_installer should be callable"
assert issubclass(Installer, Enum), "Installer should be an Enum subclass"
assert is_dataclass(InstallerInfo), "InstallerInfo should be a dataclass"

print(f"detect_installer: {detect_installer!r}")
print(f"Installer: {Installer!r}")
print(f"InstallerInfo: {InstallerInfo!r}")
print(f"UvUpgradeStrategy: {UvUpgradeStrategy!r}")

In [ ]:
# Example 2: Enumerate the Installer enum members — they should match the documented installers.
from detect_installer import Installer

members = {member.name: member.value for member in Installer}
for name, value in members.items():
    print(f"  {name} = {value!r}")

expected = {"PIP", "UV_PIP", "UV", "UV_TOOL", "PIPX", "BREW", "CONDA", "MAMBA", "UNKNOWN"}
assert expected.issubset(set(members.keys())), f"Missing enum members: {expected - set(members.keys())}"
assert Installer.PIP.value == "pip"
assert Installer.UNKNOWN.value == "unknown"

In [ ]:
# Example 3: Detect installer for a package that is definitely installed (detect_installer itself).
from detect_installer import Installer, InstallerInfo, detect_installer

info = detect_installer("detect-installer")

print(f"info = {info!r}")
assert info is not None, "detect_installer should not return None for an installed package"
assert isinstance(info, InstallerInfo)
assert isinstance(info.installer, Installer)
assert info.upgrade_cmd is None or isinstance(info.upgrade_cmd, str)
print(f"  installer = {info.installer}")
print(f"  upgrade_cmd = {info.upgrade_cmd!r}")

In [ ]:
# Example 4: Detect for a non-installed package should return None.
from detect_installer import detect_installer

result = detect_installer("this-package-definitely-does-not-exist-xyz-12345")

print(f"result for missing package = {result!r}")
assert result is None, "Expected None for a missing package"

In [ ]:
# Example 5: uv_upgrade_strategy alters the upgrade command shape for uv-project installs.
# We can't force the detected installer, but we can verify the strategy parameter is accepted
# and that the InstallerInfo dataclass is frozen (per the @dataclass(frozen=True) decorator).
from dataclasses import FrozenInstanceError

from detect_installer import Installer, InstallerInfo, detect_installer

info_add = detect_installer("detect-installer", uv_upgrade_strategy="add")
info_lock = detect_installer("detect-installer", uv_upgrade_strategy="lock")

print(f"strategy='add'  -> {info_add!r}")
print(f"strategy='lock' -> {info_lock!r}")

# Frozen dataclass: mutation should raise.
sample = InstallerInfo(Installer.PIP, "pip install -U foo")
try:
    sample.installer = Installer.UV  # type: ignore[misc]
except (FrozenInstanceError, AttributeError) as exc:
    print(f"Frozen confirmed: {type(exc).__name__}")
else:
    raise AssertionError("InstallerInfo should be frozen")

assert sample.installer is Installer.PIP
assert sample.upgrade_cmd == "pip install -U foo"